# 实验二：房源采集与 MySQL 存储

教学站：http://www.bspider.top/anjuke/

字段顺序：title → house → address → struct → area → floor → maketime → price

In [1]:
%pip install requests beautifulsoup4 pymysql

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## 1. 请求第一页并检查

请求网页、设置编码、生成 BeautifulSoup 对象；并检查是否为验证页。

In [8]:
import csv
import time
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
BASE_URL = "http://www.bspider.top/anjuke/"
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/126.0 Safari/537.36"),
    "Accept-Language": "zh-CN,zh;q=0.9",
}
MAX_PAGES = 20        # 安全页数上限，防止无限循环
DELAY = 1.5           # 每次翻页至少等待1秒
CSV_PATH = "/home/pwuser/workspace/lab02/data/houseinfo.csv"
FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]

def fetch_soup(session, url):
    """请求当前页，检查响应内容；验证页视为结构异常。"""
    r = session.get(url, timeout=30)
    r.raise_for_status()
    r.encoding = r.apparent_encoding
    soup = BeautifulSoup(r.text, "html.parser")
    if "User_Common_Antispam" in r.text or "访问验证" in soup.title.get_text():
        raise RuntimeError(f"遇到访问验证页，停止采集：{url}")
    return soup

## 2. 解析一条房源

- `select()` 选中本页全部 `div.property`；`select_one()` 在一条房源内部做相对选择
- 属性组顺序：户型、面积、朝向、楼层、建造年代；建造年代可能缺失，取值前检查长度并按文字特征分类，防止把朝向当成楼层
- 地址由多个 span 拼接，不插入空格；单价取 `property-price-average`（不是总价）

In [6]:
def parse_row(row):
    """解析一条房源，返回按 FIELDS 顺序的列表。"""
    # 1. 标题：房源容器内的 h3（相对选择器）
    node = row.select_one("h3")
    title = node.get_text(strip=True) if node else ""

    # 2. 楼盘
    node = row.select_one("p.property-content-info-comm-name")
    house = node.get_text(strip=True) if node else ""

    # 3. 地址：文字分散在多个 span 中，直接拼接，不插入空格
    node = row.select_one("p.property-content-info-comm-address")
    address = "".join(s.get_text(strip=True) for s in node.select("span")) if node else ""

    # 4-7. 户型/面积/楼层/建造年代：属性组 p 列表
    #      顺序固定为 户型、面积、朝向、楼层、建造年代，但建造年代可能缺失，
    #      因此按下标取值前必须检查长度，并按文字特征分类，防止把朝向当成楼层
    info = row.select_one("div.property-content-info:not(.property-content-info-comm)")
    ps = info.select("p.property-content-info-text") if info else []

    struct = ps[0].get_text(strip=True) if len(ps) > 0 else ""   # 第1项：户型，如 3室2厅1卫
    area = ps[1].get_text(strip=True) if len(ps) > 1 else ""     # 第2项：面积，保留单位

    floor, maketime = "", ""
    for p in ps[2:]:                                             # 第3项起：朝向/楼层/年代
        text = p.get_text(strip=True)
        if text.endswith("建造"):
            maketime = text
        elif "层" in text:
            floor = text
        # 其余为朝向，本实验不需要

    # 8. 单价：每平方米价格（不是 property-price-total 的总价）
    node = row.select_one("p.property-price-average")
    price = node.get_text(strip=True) if node else ""

    return [title, house, address, struct, area, floor, maketime, price]

先解析第一条，逐字段对照网页：

In [9]:
rows = fetch_soup(requests.Session(), BASE_URL).select("div.property")
print(len(rows))
print(parse_row(rows[0]))

RuntimeError: 遇到访问验证页，停止采集：http://www.bspider.top/anjuke/

## 3. 分页采集

- `a.next.next-active` 是活动的下一页按钮，`get("href")` 取地址（`get_text()` 只是“下一页”文字）
- `urljoin` 把 `?page=N` 补全为完整地址
- 停止条件：地址越出教学站、地址重复、下一页按钮禁用（末页）、达到安全页数上限
- 每页之间至少等待 1 秒

In [11]:
def main():
    session = requests.Session()
    session.headers.update(HEADERS)

    current_url = BASE_URL
    visited = set()
    records = []
    page_no = 0
    stop_reason = None

    while True:
        page_no += 1
        if page_no > MAX_PAGES:                       # 达到安全上限只能记为中止
            stop_reason = f"达到安全页数上限 {MAX_PAGES}，中止（不算采集完成）"
            break

        soup = fetch_soup(session, current_url)
        rows = soup.select("div.property")
        if not rows:                                  # 空列表视为结构异常
            stop_reason = f"第{page_no}页未解析到任何房源，结构异常"
            break

        for row in rows:
            records.append(parse_row(row))
        print(f"第{page_no}页 | {current_url} | 本页{len(rows)}条 | 累计{len(records)}条")

        # ---- 找下一页 ----
        nxt = soup.select_one("a.next.next-active")   # 活动的下一页按钮
        if nxt is None:
            disabled = soup.select_one("a.next")      # 下一页按钮存在但已禁用
            if disabled is not None:
                stop_reason = f"第{page_no}页为末页：下一页按钮已禁用，采集完成"
            else:
                stop_reason = f"第{page_no}页分页区域消失，结构异常（不能算采集完成）"
            break

        # ---- 补全地址 ----
        href = nxt.get("href")                        # get_text() 是“下一页”，href 才是地址
        next_url = urljoin(current_url, href)
        print(f"    下一页 href={href} -> 补全为 {next_url}")

        # ---- 停止条件：越界 / 重复 ----
        if not next_url.startswith(BASE_URL):
            stop_reason = f"下一页地址越出教学站：{next_url}"
            break
        if next_url in visited:
            stop_reason = f"下一页地址重复：{next_url}"
            break

        visited.add(current_url)
        time.sleep(DELAY)                             # 至少等待1秒
        current_url = next_url

    print("\n==== 采集结束 ====")
    print("停止原因：", stop_reason)
    print("采集总条数（去重前）：", len(records))

    # ---- 去重：按8个字段完全相同 ----
    seen = set()
    unique_records = []
    for rec in records:
        key = tuple(rec)
        if key not in seen:
            seen.add(key)
            unique_records.append(rec)
    print(f"去重前 {len(records)} 条，去重后 {len(unique_records)} 条，"
          f"重复 {len(records) - len(unique_records)} 条")

    # ---- 保存 CSV ----
    with open(CSV_PATH, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(FIELDS)
        writer.writerows(unique_records)
    print("已保存：", CSV_PATH)

    # ---- 用 DictReader 回读核对 ----
    with open(CSV_PATH, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        assert reader.fieldnames == FIELDS, f"表头不符：{reader.fieldnames}"
        data = list(reader)
    print(f"回读数据行数（不含表头）：{len(data)}")
    for i in (0, 60, len(data) - 1):                  # 第一条、跨页一条、最后一条
        print(f"样本第{i + 1}条：", {k: data[i][k] for k in FIELDS})

In [12]:
if __name__ == "__main__":
    main()

第1页 | http://www.bspider.top/anjuke/ | 本页60条 | 累计60条
    下一页 href=?page=2 -> 补全为 http://www.bspider.top/anjuke/?page=2
第2页 | http://www.bspider.top/anjuke/?page=2 | 本页60条 | 累计120条
    下一页 href=?page=3 -> 补全为 http://www.bspider.top/anjuke/?page=3
第3页 | http://www.bspider.top/anjuke/?page=3 | 本页60条 | 累计180条
    下一页 href=?page=4 -> 补全为 http://www.bspider.top/anjuke/?page=4
第4页 | http://www.bspider.top/anjuke/?page=4 | 本页60条 | 累计240条
    下一页 href=?page=5 -> 补全为 http://www.bspider.top/anjuke/?page=5
第5页 | http://www.bspider.top/anjuke/?page=5 | 本页60条 | 累计300条
    下一页 href=?page=6 -> 补全为 http://www.bspider.top/anjuke/?page=6
第6页 | http://www.bspider.top/anjuke/?page=6 | 本页60条 | 累计360条
    下一页 href=?page=7 -> 补全为 http://www.bspider.top/anjuke/?page=7
第7页 | http://www.bspider.top/anjuke/?page=7 | 本页60条 | 累计420条
    下一页 href=?page=8 -> 补全为 http://www.bspider.top/anjuke/?page=8
第8页 | http://www.bspider.top/anjuke/?page=8 | 本页60条 | 累计480条
    下一页 href=?page=9 -> 补全为 http://www.bspider.top/anjuke/

## 4. 回读核对

运行完上面的采集后，用 `csv.DictReader` 重新读取文件核对：

In [13]:
import csv

with open(CSV_PATH, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    assert reader.fieldnames == FIELDS, f"表头不符：{reader.fieldnames}"
    data = list(reader)
print("数据行数（不含表头）：", len(data))
for i in (0, 60, len(data) - 1):
    print(f"第{i + 1}条：", {k: data[i][k] for k in FIELDS})

数据行数（不含表头）： 498
第1条： {'title': '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', 'house': '万科金域学府翰林菁华里', 'address': '南岸学府大道二塘路59号', 'struct': '3室2厅1卫', 'area': '89.9㎡', 'floor': '低层(共27层)', 'maketime': '2019年建造', 'price': '16465元/㎡'}
第61条： {'title': '万科物业自营 朝中庭 业主诚心出售', 'house': '万科金域学府翰林菁华里', 'address': '南岸学府大道二塘路59号', 'struct': '3室2厅1卫', 'area': '85.7㎡', 'floor': '低层(共27层)', 'maketime': '2018年建造', 'price': '16914元/㎡'}
第498条： {'title': '恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售', 'house': '恒大照母山山水城', 'address': '渝北照母山恒康路9号', 'struct': '3室2厅1卫', 'area': '90.2㎡', 'floor': '低层(共32层)', 'maketime': '2015年建造', 'price': '15530元/㎡'}


---

## 第三阶段：MySQL 数据存储

读取第二阶段保存的 `data/houseinfo.csv` 写入 MySQL，不重新爬取网页。

连接参数：host=mysql，port=3306，database=smartpark_lab，user=student，charset=utf8mb4，autocommit=False。

In [14]:
# -*- coding: utf-8 -*-
"""
实验二 第三阶段：把 data/houseinfo.csv 写入 MySQL（不重新采集）
要点：pymysql 连接 → executemany 批量插入 → commit / rollback → 关闭游标和连接
     按8个字段比较库内已有记录，只插入不存在的记录，避免重复运行时重复插入
"""
import csv

import pymysql

CSV_PATH = "/home/pwuser/workspace/lab02/data/houseinfo.csv"
FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]

# ---- 1. 读取CSV，检查表头，按8个字段固定顺序整理记录 ----
with open(CSV_PATH, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    assert reader.fieldnames == FIELDS, f"表头不符：{reader.fieldnames}"
    records = [tuple(row[k] for k in FIELDS) for row in reader]
print(f"CSV数据行数：{len(records)}，每条{len(records[0])}项")

conn = None
cursor = None
try:
    # ---- 2. 连接数据库（autocommit=False，由程序提交事务）----
    conn = pymysql.connect(
        host="mysql", port=3306,
        user="student", password="123456",
        database="smartpark_lab", charset="utf8mb4",
        autocommit=False,
    )
    cursor = conn.cursor()

    # 连接正常性检查：查询一条数据
    cursor.execute("SELECT COUNT(*) FROM houseinfo")
    before_count = cursor.fetchone()[0]
    print(f"入库前 houseinfo 条数：{before_count}")

    # ---- 3. 查询库内已有记录，避免重复插入 ----
    cursor.execute(
        "SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo"
    )
    existing = set(cursor.fetchall())
    new_records = [r for r in records if r not in existing]
    print(f"与库内比对后需要新插入：{len(new_records)} 条（跳过已存在 {len(records) - len(new_records)} 条）")

    # ---- 4. INSERT：8个字段对应8个 %s 参数位置 ----
    sql = ("INSERT INTO houseinfo "
           "(title, house, address, struct, area, floor, maketime, price) "
           "VALUES (%s, %s, %s, %s, %s, %s, %s, %s)")

    if new_records:
        # ---- 5. 批量写入 ----
        affected = cursor.executemany(sql, new_records)
        print(f"executemany 受影响行数：{affected}")
        conn.commit()                              # 写入成功后提交
        print("已提交事务 commit()")
    else:
        print("没有新记录需要插入，不执行写入")

    cursor.execute("SELECT COUNT(*) FROM houseinfo")
    after_count = cursor.fetchone()[0]
    print(f"入库后 houseinfo 条数：{after_count}，本次新增：{after_count - before_count}")

except Exception as e:
    print("出错：", repr(e))
    if conn is not None:
        conn.rollback()                            # 写入出错时回滚，撤销未提交修改
        print("已回滚事务 rollback()")
finally:
    # ---- 6. 无论成功失败都释放资源：先游标后连接 ----
    if cursor is not None:
        cursor.close()
    if conn is not None:
        conn.close()
    print("游标和连接已关闭")

CSV数据行数：498，每条8项
入库前 houseinfo 条数：498
与库内比对后需要新插入：0 条（跳过已存在 498 条）
没有新记录需要插入，不执行写入
入库后 houseinfo 条数：498，本次新增：0
游标和连接已关闭


再次运行一次，验证重跑不会重复插入（应显示需要新插入 0 条）：

In [1]:
# -*- coding: utf-8 -*-
"""第三阶段核对：CSV 第1、61、498条 与数据库 id=1、61、498 逐字段比较。"""
import csv

import pymysql

CSV_PATH = "/home/pwuser/workspace/lab02/data/houseinfo.csv"
FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]

with open(CSV_PATH, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    assert reader.fieldnames == FIELDS
    csv_rows = list(reader)

conn = pymysql.connect(host="mysql", port=3306, user="student", password="123456",
                       database="smartpark_lab", charset="utf8mb4")
try:
    with conn.cursor() as cur:
        cur.execute("SELECT id,title,house,address,struct,area,floor,maketime,price "
                    "FROM houseinfo WHERE id IN (1,61,498) ORDER BY id")
        db_rows = cur.fetchall()
finally:
    conn.close()

for idx, db in zip((0, 60, len(csv_rows) - 1), db_rows):
    c = csv_rows[idx]
    d = db[1:]
    ok = all(c[k] == d[j] for j, k in enumerate(FIELDS))
    print(f"id={db[0]} CSV第{idx + 1}条 逐字段一致: {ok}")
    for j, k in enumerate(FIELDS):
        mark = "OK " if c[k] == d[j] else "DIFF"
        print(f"  [{mark}] {k}: csv={c[k]!r} db={d[j]!r}")

id=1 CSV第1条 逐字段一致: True
  [OK ] title: csv='万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器' db='万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器'
  [OK ] house: csv='万科金域学府翰林菁华里' db='万科金域学府翰林菁华里'
  [OK ] address: csv='南岸学府大道二塘路59号' db='南岸学府大道二塘路59号'
  [OK ] struct: csv='3室2厅1卫' db='3室2厅1卫'
  [OK ] area: csv='89.9㎡' db='89.9㎡'
  [OK ] floor: csv='低层(共27层)' db='低层(共27层)'
  [OK ] maketime: csv='2019年建造' db='2019年建造'
  [OK ] price: csv='16465元/㎡' db='16465元/㎡'
id=61 CSV第61条 逐字段一致: True
  [OK ] title: csv='万科物业自营 朝中庭 业主诚心出售' db='万科物业自营 朝中庭 业主诚心出售'
  [OK ] house: csv='万科金域学府翰林菁华里' db='万科金域学府翰林菁华里'
  [OK ] address: csv='南岸学府大道二塘路59号' db='南岸学府大道二塘路59号'
  [OK ] struct: csv='3室2厅1卫' db='3室2厅1卫'
  [OK ] area: csv='85.7㎡' db='85.7㎡'
  [OK ] floor: csv='低层(共27层)' db='低层(共27层)'
  [OK ] maketime: csv='2018年建造' db='2018年建造'
  [OK ] price: csv='16914元/㎡' db='16914元/㎡'
id=498 CSV第498条 逐字段一致: True
  [OK ] title: csv='恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售' db='恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售'
  [OK ] house: csv='恒大照母山山水城' db='恒大照母山山水城'
  [OK ] ad